In [ ]:
# This notebook is meant to generate the pedestal file used in simulation, taking the
# pedestal file used for data as input. Because there are tile swaps in data, which is
# currently not taken care of in simulation, we have to manually swap the tile labels
# in simulation so that the channel-by-channel pedestals correspond to the same physical space. 

In [ ]:
import json
import numpy as np

In [ ]:
with open('/global/common/software/dune/mkramer/devel/flow4pedestal/reference-cold-pedestal-2024_06_05_08_28_19_CDTevd_ped.tile_id.decimal.json', 'r') as f:
    data = json.load(f)

In [ ]:
# flow uses a "global" indexing for the tile number, so that module 0 has tiles 1-16
# module 1 has 17-32, etc. 
# The geometry doc, on the other hand, labels the tiles 1-16 for each module.
# This function just does a conversion of "local" indexing to "global" indexing
def global_tile_id(module, tile):
    return (module) * 16 + tile

In [ ]:
# These numbers were obtained from Figure 13 and 14 of the charge geometry document
# https://docs.dunescience.org/cgi-bin/sso/RetrieveFile?docid=32440&filename=2x2_Demonstrator_Geometry_Description-2.pdf&version=2
# (x,y) corresponds to the tile in the "local" indexing, where x is the module number and
# y is the tile number. 
swaps = {}
swaps[(0,4)] = (0,8)
swaps[(0,7)] = (0,4)
swaps[(0,8)] = (0,7)


swaps[(1,9)] = (1,10)
swaps[(1,10)] = (1,9)

swaps[(2,7)] = (2,8)
swaps[(2,8)] = (2,7)

swaps[(3,5)] = (3,8)
swaps[(3,8)] = (3,5)
swaps[(3,9)] = (3,10)
swaps[(3,10)] = (3,9)

global_swaps = {}
for k, v in swaps.items():
    old = "{:02d}".format(global_tile_id(*k))
    new = "{:02d}".format(global_tile_id(*v))
    global_swaps[old] = new

In [ ]:
global_swaps

In [ ]:
# Probably not the best way to do this, but I do the replacement by taking the string position
# corresponding to the tile number and manipulating that
swapped_dict = {}
for key, value in data.items():
    old_tile = key[3:5]
    if old_tile in global_swaps.keys():
        new_tile = global_swaps[old_tile]
        new_key = key[:3] + new_tile + key[5:]
        print("Old key: {old_key}\nNew key: {new_key}\n".format(old_key = key, new_key=new_key))
    else:
        new_key = key

    swapped_dict[new_key] = value

In [ ]:
with open('reference-cold-pedestal-2024_06_05_08_28_19_CDTevd_ped.tile_id_swapped.decimal.json', 'w') as f:
    json.dump(swapped_dict, f)